# Lección 4.4 — Sobreajuste vs. subajuste: memorizar no es aprender

> **Módulo 4 · Evaluación: ¿mi modelo es bueno?** · Nivel 2 — Aprendiz
> ⏱️ Tiempo estimado: 90–110 min (lección con ejercicios intercalados ≈ 65 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lección 2.5 (entrenamiento, validación y prueba), Lección 3.7 (curvas de aprendizaje), Lección 4.1 (RMSE)

### 🎯 Objetivos
Al terminar esta lección podrás:
- Explicar con una analogía la diferencia entre **sobreajuste** (*overfitting*) y **subajuste** (*underfitting*).
- Diagnosticar cuál de los dos tiene un modelo comparando el error de **entrenamiento** con el de **validación**.
- Leer una **curva de validación** (error vs. complejidad) y una **curva de aprendizaje** (error vs. cantidad de datos).
- Elegir el remedio adecuado para cada caso.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación
Ejecuta esta celda una vez (`Shift + Enter`). Carga las herramientas que usaremos.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error

def rmse(y_real, y_pred):                     # el RMSE de la Lección 4.1
    return np.sqrt(mean_squared_error(y_real, y_pred))

plt.rcParams["figure.figsize"] = (8, 4.5)
print("✅ Todo listo")

---
## 1. 📖 La intuición: tres estudiantes y un examen

El profesor reparte una **guía de práctica** con 50 preguntas resueltas. El examen real tendrá preguntas **parecidas, pero no iguales**.

| Estudiante | Cómo se prepara | Guía de práctica | Examen real |
|---|---|---|---|
| 🦜 **Memo** | Se aprende de memoria las 50 respuestas, incluso las erratas de la guía | 100 % | 55 % |
| 😴 **Flojo** | Lee solo los títulos: "todo se resuelve con una regla de tres" | 60 % | 58 % |
| 🧠 **Comprende** | Entiende el *porqué* de cada solución | 92 % | 88 % |

- **Memo** se **sobreajustó**: aprendió el ruido de la guía (hasta las erratas) y no sabe generalizar.
- **Flojo** se **subajustó**: su idea es demasiado simple; le va mal en todos lados.
- **Comprende** captó el **patrón**: le va casi igual de bien en la guía que en el examen.

Un modelo de ML vive exactamente lo mismo: los datos de **entrenamiento** son la guía; los datos de **validación/prueba**, el examen.

### ✋ Ejercicio 1 — ¿Quién es quién?
Tres modelos predicen el precio de casas. Error (RMSE) en miles de pesos:

| Modelo | Entrenamiento | Validación |
|---|---|---|
| A | 210 | 215 |
| B | 3 | 190 |
| C | 60 | 70 |

¿Cuál es Memo, cuál es Flojo y cuál es Comprende?

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

- **B = Memo (sobreajuste):** error casi cero en entrenamiento y enorme en validación. La **brecha** es la firma del sobreajuste.
- **A = Flojo (subajuste):** error alto **en los dos**, sin brecha. El modelo ni siquiera aprende bien los datos que ya vio.
- **C = Comprende:** error bajo en los dos y brecha pequeña.

Regla de oro: **compara siempre los dos errores**. Con uno solo no puedes diagnosticar nada.
</details>

---
## 2. 📖 El concepto: el patrón y el ruido

Todo conjunto de datos tiene dos ingredientes:
- **Patrón** (señal): la relación real que se repite y que queremos aprender.
- **Ruido:** las variaciones al azar de cada ejemplo, que **no** se repiten.

| | Qué aprende | Error de entrenamiento | Error de validación |
|---|---|---|---|
| **Subajuste** | Menos que el patrón | Alto | Alto (parecido al de entrenamiento) |
| **Buen ajuste** | El patrón | Bajo | Bajo (un poco mayor) |
| **Sobreajuste** | El patrón **y el ruido** | Muy bajo | Alto (**gran brecha**) |

Veámoslo con datos. Una tienda registró cuántos clientes hay dentro según las horas desde que abre. Solo tenemos **15 registros** para entrenar, y otros 300 que guardamos para validar.

In [ ]:
def generar_datos(n, semilla):
    rng = np.random.default_rng(semilla)
    horas = np.sort(rng.uniform(0, 10, n))
    patron = 20 + 8 * np.sin(horas / 1.6)                 # la relación real (el modelo NO la conoce)
    clientes = patron + rng.normal(0, 2.5, n)              # + ruido: lo que varía al azar cada día
    return horas.reshape(-1, 1), clientes

X_tr, y_tr = generar_datos(15, semilla=1)      # entrenamiento: 15 registros
X_va, y_va = generar_datos(300, semilla=100)   # validación: 300 registros que el modelo no verá al entrenar

plt.scatter(X_tr, y_tr, s=40, edgecolor="k", label="Entrenamiento (15)")
plt.scatter(X_va, y_va, s=8, alpha=.3, label="Validación (300)")
plt.xlabel("Horas desde la apertura"); plt.ylabel("Clientes en la tienda"); plt.legend(); plt.show()

### ✋ Ejercicio 2 — ¿Qué forma tiene el patrón?
Mirando la nube de puntos de validación: ¿una **recta** podría describir la relación? Dibuja en el aire (o en papel) la curva que tú usarías.

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

No: los clientes **suben** hasta un pico (≈ 2–3 h después de abrir), **bajan** hasta un valle (≈ 7–8 h) y vuelven a subir. Una recta no puede seguir esas curvas: será un modelo "Flojo". Necesitamos algo más flexible… pero no **demasiado**.
</details>

---
## 3. 📖 Cómo se ve: la perilla de la complejidad

Usaremos **regresión polinomial**: una línea que puede curvarse. Su perilla es el **grado** del polinomio:
- **Grado 1:** una recta.
- **Grado 3:** una curva con un par de "jorobas".
- **Grado 12:** una curva que puede retorcerse por todas partes.

Más grado = **más complejidad** = más libertad para acomodarse a los puntos.

### 🔮 Ejercicio 3 — Predice antes de ejecutar
Vamos a entrenar los grados **1, 3 y 12** con los 15 puntos.
1. ¿Cuál tendrá el **menor error en entrenamiento**?
2. ¿Cuál tendrá el **menor error en validación**?

✍️ *Mis predicciones:*

In [ ]:
def polinomio(grado):
    return make_pipeline(PolynomialFeatures(grado), StandardScaler(), LinearRegression())

x_linea = np.linspace(0, 10, 400).reshape(-1, 1)
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8), sharey=True)
for ax, grado in zip(axes, [1, 3, 12]):
    m = polinomio(grado).fit(X_tr, y_tr)
    ax.scatter(X_va, y_va, s=6, alpha=.2)
    ax.scatter(X_tr, y_tr, s=35, edgecolor="k")
    ax.plot(x_linea, m.predict(x_linea), "r", lw=2)
    ax.set_ylim(5, 35)
    ax.set_title(f"Grado {grado}\nRMSE entrenamiento {rmse(y_tr, m.predict(X_tr)):.2f} · validación {rmse(y_va, m.predict(X_va)):.2f}", fontsize=10)
plt.tight_layout(); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

| Grado | Entrenamiento | Validación | Diagnóstico |
|---|---|---|---|
| 1 | 4.43 | 4.39 | 😴 **Subajuste**: la recta ignora las curvas; error alto en los dos |
| 3 | 2.38 | 2.91 | 🧠 **Buen ajuste**: sigue el patrón; brecha pequeña |
| 12 | 1.85 | **29.44** | 🦜 **Sobreajuste**: pasa cerca de cada punto y se vuelve loca entre ellos |

1. El **grado 12**: con tanta libertad, se acomoda a cada punto de entrenamiento, ruido incluido.
2. El **grado 3**. El grado 12, el mejor en entrenamiento, es por mucho el **peor** en validación: ¡10 veces peor que el grado 3!

> 📝 El ruido que metimos tiene una desviación de 2.5. Ningún modelo puede bajar de ~2.5 en validación: es el **error irreducible**. Un error de entrenamiento **por debajo** de ese piso (como el 1.85 del grado 12) ya es una pista de que el modelo está memorizando ruido.
</details>

---
## 4. 📖 La curva de validación: buscar el fondo de la "U"

En lugar de probar tres grados, probemos **todos** del 1 al 12 y dibujemos los dos errores. A esta gráfica (error vs. complejidad) se le llama **curva de validación** o **curva de complejidad**.

### 🔮 Ejercicio 4 — Dibuja antes de ver
Antes de ejecutar, dibuja en papel cómo crees que se verán:
- la curva del error de **entrenamiento** al subir el grado;
- la curva del error de **validación** al subir el grado.

✍️ *Mi dibujo / descripción:*

In [ ]:
GRADOS = range(1, 13)
err_tr, err_va = [], []
for g in GRADOS:
    m = polinomio(g).fit(X_tr, y_tr)
    err_tr.append(rmse(y_tr, m.predict(X_tr)))
    err_va.append(rmse(y_va, m.predict(X_va)))

plt.plot(GRADOS, err_tr, "o-", label="Entrenamiento")
plt.plot(GRADOS, err_va, "s-", label="Validación")
plt.axhline(2.5, color="gray", ls=":", label="Ruido (error irreducible)")
plt.ylim(0, 8); plt.xlabel("Grado del polinomio (complejidad →)"); plt.ylabel("RMSE")
plt.text(1.2, 7.3, "← subajuste"); plt.text(9.3, 7.3, "sobreajuste →")
plt.legend(loc="center right"); plt.grid(alpha=.3); plt.title("Curva de validación"); plt.show()

for g, a, b in zip(GRADOS, err_tr, err_va):
    print(f"grado {g:>2}: entrenamiento {a:5.2f} · validación {b:6.2f}")

<details><summary><b>👀 Ver respuesta</b></summary>

- **Entrenamiento:** baja (casi) siempre. Más complejidad = más capacidad de acomodarse a los puntos que ya vio. (Va de 4.43 a ~1.8.)
- **Validación:** baja, toca fondo y vuelve a subir: una **"U"**. El fondo está en el **grado 3** (2.91). A partir del grado 6 sube, y desde el 10 se dispara (≈ 29, fuera de la gráfica).

La zona buena es el **fondo de la U de validación**. A la izquierda: subajuste. A la derecha: sobreajuste.

> ⚠️ Fíjate en que la curva no es perfectamente suave (el grado 4 sale un poco peor que el 3 y el 5). Con solo 15 puntos de entrenamiento hay mucho azar. Ante un empate, elige el modelo **más simple**.
</details>

### ✋ Ejercicio 5 — Lee la curva
1. Si solo miraras la curva de **entrenamiento**, ¿qué grado elegirías? ¿Qué pasaría en producción?
2. Entre el grado 3 y el 5, la validación es casi igual (2.91 vs 3.04). ¿Cuál eliges y por qué?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. Elegirías el grado más alto (10 o más), porque es el que menos error tiene en entrenamiento… y en producción tendrías un error de ~29 clientes: **un desastre**. Por eso **nunca** se elige un modelo mirando el error de entrenamiento.
2. El **grado 3**: es algo mejor y además **más simple**. Un modelo simple es más estable, más fácil de explicar y tiene menos riesgo de sobreajustar con datos nuevos (es la "navaja de Ockham").
</details>

---
## 5. 📖 ¿Qué pasa si consigo más datos?

La pregunta del millón en cualquier empresa: *"¿Compramos/etiquetamos más datos?"*. La respuesta depende del diagnóstico.

Una **curva de aprendizaje** dibuja el error contra la **cantidad de datos de entrenamiento**. Ya viste curvas de aprendizaje contra las **épocas** en la Lección 3.7; esta es su hermana: misma idea (entrenamiento vs. validación), otro eje horizontal.

### 🔮 Ejercicio 6 — ¿A quién le ayudan más datos?
Vamos a entrenar el **grado 1** (subajuste) y el **grado 12** (sobreajuste) con 15, 30, 50, 100, 200, 500 y 1,000 registros.
- ¿Cuál de los dos mejorará mucho en validación?
- ¿Cuál se quedará estancado aunque le des 1,000 registros?

✍️ *Mis predicciones:*

In [ ]:
TAMANOS = [15, 30, 50, 100, 200, 500, 1000]
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), sharey=True)
for ax, grado in zip(axes, [1, 12]):
    tr, va = [], []
    for n in TAMANOS:
        Xn, yn = generar_datos(n, semilla=1)
        m = polinomio(grado).fit(Xn, yn)
        tr.append(rmse(yn, m.predict(Xn))); va.append(rmse(y_va, m.predict(X_va)))
    ax.plot(TAMANOS, tr, "o-", label="Entrenamiento"); ax.plot(TAMANOS, va, "s-", label="Validación")
    ax.axhline(2.5, color="gray", ls=":"); ax.set_xscale("log"); ax.set_ylim(0, 8)
    ax.set_xlabel("Registros de entrenamiento"); ax.set_title(f"Grado {grado}"); ax.legend(); ax.grid(alpha=.3)
    print(f"Grado {grado:>2}: validación con 15 → {va[0]:.2f}  ·  con 1,000 → {va[-1]:.2f}")
axes[0].set_ylabel("RMSE"); plt.tight_layout(); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

- **Grado 12 (sobreajuste):** con 15 registros su error de validación es enorme (29.4, fuera de la gráfica), pero con 1,000 baja a **2.55**, casi el piso del ruido. Con muchos datos, ya no puede "acomodarse" a cada punto: el ruido se promedia. Las dos curvas **se juntan**.
- **Grado 1 (subajuste):** se queda en **≈ 4.4** sin importar cuántos datos le des. Las curvas se juntan **rápido, pero arriba**. Una recta no puede dibujar curvas, ni con un millón de puntos.

**Moraleja:**
- Brecha grande que se cierra con más datos → **sobreajuste** → más datos **sí** ayudan.
- Curvas juntas pero altas → **subajuste** → más datos **no** ayudan; necesitas un modelo más flexible o mejores características (Lección 2.4).
</details>

---
## 6. 📖 Perillas de complejidad: el mismo dilema en todos los modelos

Cada modelo tiene su propia perilla de complejidad, pero el dilema es siempre el mismo:

| Modelo | Perilla | Para **más** complejidad (→ riesgo de sobreajuste) | Para **menos** complejidad (→ riesgo de subajuste) |
|---|---|---|---|
| Regresión polinomial | Grado | Subir el grado | Bajar el grado |
| Árbol de decisión (5.4) | `max_depth`, `min_samples_leaf` | Subir `max_depth`, bajar `min_samples_leaf` | Bajar `max_depth`, subir `min_samples_leaf` |
| KNN (5.3) | `n_neighbors` (k) | **Bajar** k | **Subir** k |
| Redes neuronales (M9) | Capas, neuronas, épocas | Más capas/neuronas/épocas | Menos; *early stopping* |
| Casi todos (M7) | Regularización (`alpha`, `C`…) | Menos regularización | Más regularización |

> 🧠 No existe un "valor correcto" universal para estas perillas: se elige buscando el fondo de la "U" en **validación** (mejor aún, con validación cruzada, Lección 4.6).

### ✏️ Ejercicio 7 — Juega con dos perillas a la vez
Cambia `GRADO` y `N_DATOS` y observa el dibujo. Intenta:
1. `GRADO = 12`, `N_DATOS = 15` → ¿qué ves?
2. `GRADO = 12`, `N_DATOS = 500` → ¿y ahora?
3. `GRADO = 1`, `N_DATOS = 500`.

In [ ]:
GRADO = 12       # ✏️
N_DATOS = 15     # ✏️

Xn, yn = generar_datos(N_DATOS, semilla=1)
m = polinomio(GRADO).fit(Xn, yn)
plt.scatter(Xn, yn, s=12, edgecolor="k", alpha=.6)
plt.plot(x_linea, m.predict(x_linea), "r", lw=2); plt.ylim(5, 35)
plt.title(f"Grado {GRADO} con {N_DATOS} datos · RMSE validación = {rmse(y_va, m.predict(X_va)):.2f}"); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

1. Grado 12 con 15 datos: la curva hace picos y caídas absurdos entre los puntos (validación ≈ 29). **Sobreajuste.**
2. Grado 12 con 500 datos: la misma "curva loca" ahora se ve suave y sigue el patrón (validación ≈ 2.6). Los datos **domaron** la complejidad.
3. Grado 1 con 500 datos: sigue siendo una recta (validación ≈ 4.4). **Subajuste**, por muchos datos que haya.

La complejidad "adecuada" **depende de cuántos datos tienes**: con pocos datos, modelos simples; con muchos, puedes permitirte modelos más flexibles.
</details>

---
## 7. 📖 Diagnóstico y remedios

| 🚨 Síntoma | Diagnóstico | Qué probar |
|---|---|---|
| Error de entrenamiento muy bajo, validación mucho peor (gran brecha) | **Sobreajuste** | Más datos · modelo más simple (bajar grado/`max_depth`, subir k) · regularización (Lección 7.1) · *early stopping* (Lección 7.2) · quitar características inútiles |
| Error alto en entrenamiento **y** validación, parecidos | **Subajuste** | Modelo más flexible · mejores características (Lección 2.4) · menos regularización · entrenar más tiempo (Lección 3.4) |
| Error bajo en los dos, brecha pequeña | **Buen ajuste** 🎉 | Confirmar una sola vez con el conjunto de prueba |
| Validación **mejor** que entrenamiento, o resultados "demasiado buenos" | Algo raro: fuga de datos, validación muy pequeña o fácil | Revisar fugas (Lección 2.6) y cómo se dividieron los datos |

### 🔮 Ejercicio 8 — Sé el doctor
Para cada caso, da el diagnóstico y **un** remedio:
1. Exactitud de un clasificador: entrenamiento 99 %, validación 74 %.
2. RMSE de un modelo de precios: entrenamiento 48, validación 50. La línea base de "predecir el promedio" (Lección 4.7) tiene 52.
3. Una red neuronal: el error de validación bajó durante 20 épocas y desde entonces sube, mientras el de entrenamiento sigue bajando.

✍️ *Mis diagnósticos:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. **Sobreajuste** (brecha de 25 puntos). Simplificar el modelo, regularizar o conseguir más datos.
2. **Subajuste**: no hay brecha, pero el modelo apenas le gana a predecir el promedio. Un modelo más flexible o mejores características.
3. **Sobreajuste que empieza en la época 20.** Es la curva de la Lección 3.7. Remedio: ***early stopping***, quedarse con el modelo de la época 20 (Lección 7.2).
</details>

---
## 8. 📖 Errores comunes al diagnosticar

| ❌ Error | Por qué es un problema | ✅ Qué hacer |
|---|---|---|
| Reportar el error de **entrenamiento** como "el desempeño del modelo" | Premia la memorización; el grado 12 parecería el mejor | Reportar siempre validación o prueba |
| Elegir la complejidad mirando el conjunto de **prueba** | La prueba deja de ser un examen sorpresa: el resultado sale optimista | Elegir con validación (o validación cruzada, Lección 4.6); prueba una sola vez al final |
| Concluir "sobreajuste" con un conjunto de validación diminuto | Con pocos datos la validación es muy ruidosa | Validación cruzada |
| Pedir más datos para un modelo con subajuste | No ayudará (Ejercicio 6) | Cambiar de modelo o de características |
| Creer que "más complejo siempre es mejor" | Solo es cierto en entrenamiento | Buscar el fondo de la "U" |

### ✋ Ejercicio 9 — El atajo tentador
Un compañero prueba 40 grados distintos, mira el error en el **conjunto de prueba** de cada uno, se queda con el mejor y reporta ese error. ¿Por qué ese número es demasiado optimista?

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Porque al elegir entre 40 opciones **usando la prueba**, eligió también la que tuvo más **suerte** con esos datos concretos. La prueba se convirtió en parte del entrenamiento (sirvió para decidir algo) y ya no mide cómo le irá con datos realmente nuevos. Es como estudiar con el examen real y luego presumir la calificación. Se elige con validación; la prueba se usa **una sola vez**.
</details>

---
## 9. 🧠 Resumen en 5 puntos
1. **Subajuste:** el modelo es demasiado simple; error alto en entrenamiento **y** validación.
2. **Sobreajuste:** el modelo memoriza el ruido; error bajo en entrenamiento y **mucho** mayor en validación (gran **brecha**).
3. Al subir la complejidad, el error de entrenamiento baja siempre, pero el de validación forma una **"U"**: el mejor modelo está en el fondo.
4. Las **curvas de aprendizaje** dicen si más datos ayudarán: sí con sobreajuste, no con subajuste.
5. Se diagnostica y se elige con **validación**; el conjunto de prueba se usa una sola vez, al final.

---
## 10. 🏋️ Ejercicios finales

Ya practicaste cada idea por separado. Ahora toca **combinarlas**. Estos ejercicios son más largos (≈30 min).

### 10.1 Repaso rápido (opción múltiple)

**1.** Entrenamiento: RMSE 2.0. Validación: RMSE 2.2. El error irreducible del problema es ~2. ¿Diagnóstico?
- a) Sobreajuste · b) Subajuste · c) Buen ajuste · d) Fuga de datos segura

**2.** ¿Qué remedio **no** sirve contra el sobreajuste?
- a) Conseguir más datos
- b) Subir el grado del polinomio
- c) Regularizar
- d) Bajar `max_depth` del árbol

**3.** En KNN, ¿qué valor de k tiene más riesgo de **sobreajuste**?
- a) k = 1 · b) k = 50 · c) k = 500 · d) Todos igual

**4.** Tu modelo tiene subajuste. Tu jefe ofrece pagar para etiquetar el doble de datos. ¿Qué le dices?
- a) Sí, más datos siempre ayudan
- b) Primero probemos un modelo más flexible o mejores características
- c) Hay que bajar la complejidad
- d) Hay que usar el conjunto de prueba para ajustar

**5.** En la curva de validación, el error de entrenamiento…
- a) Forma una "U" · b) Suele bajar al subir la complejidad · c) Sube al subir la complejidad · d) No cambia

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **c)** Los dos errores están cerca del piso del ruido y la brecha es pequeña.
2. **b)** Subir el grado **aumenta** la complejidad: empeora el sobreajuste.
3. **a)** Con k = 1, cada predicción copia al vecino más cercano, ruido incluido: memoriza.
4. **b)** Con subajuste las curvas de aprendizaje ya están juntas y altas: más datos no las bajan.
5. **b)** Más complejidad = más capacidad de acomodarse a los datos vistos. La "U" es la curva de **validación**.
</details>

### 10.2 Preguntas para pensar (sin ejecutar)

**A.** ¿Por qué decimos que un modelo sobreajustado "aprendió el ruido"? Usa el ejemplo de la tienda: ¿qué sería el ruido allí?

**B. Caso de análisis 💼** Un equipo entrena un modelo para predecir la demanda de un producto. Resultados: entrenamiento con error de 5 %, validación con error de 6 %, pero, tras un mes en producción, el error real es de **18 %**. Ni subajuste ni sobreajuste explican esto a simple vista. ¿Qué pudo pasar? Da dos hipótesis.

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** El ruido son las variaciones de un día concreto que **no se repetirán**: un día llovió, otro hubo un evento en la calle, alguien contó mal. El grado 12 dobló su curva para pasar por esos puntos como si fueran reglas. Con datos nuevos, esas "reglas" no se cumplen, y se equivoca.

**B.** Algunas hipótesis razonables:
- **La validación no se parecía a producción**: por ejemplo, se mezclaron datos de fechas al azar y el modelo "vio el futuro" (en series de tiempo hay que validar respetando el orden temporal, Lección 4.6).
- **Fuga de datos** (Lección 2.6): alguna característica no está disponible igual en producción.
- **El mundo cambió** (deriva de datos, Lección 15.2): nuevos competidores, otra temporada, un cambio de precios.
Lección: un buen diagnóstico entrenamiento/validación es **necesario, pero no suficiente**: la validación debe imitar cómo se usará el modelo.
</details>

### 10.3 🧩 Práctica integradora: el modelo que "promedia vecinos"

Un hospital quiere predecir qué tanto **avanzará la diabetes** de un paciente en un año (un número: más alto = peor), a partir de 10 mediciones (edad, índice de masa corporal, presión arterial, análisis de sangre…). Son datos reales de 442 pacientes.

Usarás **KNN para regresión** (lo verás a fondo en la Lección 5.3): para predecir a un paciente, busca a los **k pacientes más parecidos** del entrenamiento y promedia su avance. Su perilla de complejidad es **k**.

Lo harás en 6 pasos cortos. Cada uno tiene su respuesta: **no avances sin revisarla**.

**Paso 1 ▶️ — Datos.** Guardamos un 20 % como prueba, bajo llave. Para validar usaremos **validación cruzada de 5 pliegues** sobre el entrenamiento (un adelanto de la Lección 4.6: es como repetir la validación 5 veces con trozos distintos y promediar).

In [ ]:
Xd, yd = load_diabetes(return_X_y=True)
Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(Xd, yd, test_size=0.2, random_state=0)
print(f"Entrenamiento: {len(yd_tr)} pacientes · Prueba: {len(yd_te)} pacientes")

def evaluar_knn(k):
    modelo = make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=k))
    error_tr = rmse(yd_tr, modelo.fit(Xd_tr, yd_tr).predict(Xd_tr))
    error_cv = -cross_val_score(modelo, Xd_tr, yd_tr, cv=5, scoring="neg_root_mean_squared_error").mean()
    return error_tr, error_cv

**Paso 2 🔮 — k = 1.** Con k = 1, cada paciente de entrenamiento tiene como vecino más cercano… a sí mismo. Predice: ¿qué RMSE tendrá en **entrenamiento**? ¿Y en validación cruzada?

✍️ *Mi apuesta:*

In [ ]:
tr, cv = evaluar_knn(1)
print(f"k=1 → RMSE entrenamiento {tr:.1f} · validación cruzada {cv:.1f}")

<details><summary><b>👀 Ver respuesta</b></summary>

Entrenamiento **0.0** (cada paciente se "predice" copiándose a sí mismo) y validación cruzada **76.8**. Brecha gigantesca: **sobreajuste** de manual. Un error de 0 en entrenamiento no es un logro: es memoria.
</details>

**Paso 3 🔮 — El otro extremo.** Ahora k = 250: cada predicción promedia a casi todos los pacientes de entrenamiento. ¿Qué diagnóstico esperas?

In [ ]:
tr, cv = evaluar_knn(250)
print(f"k=250 → RMSE entrenamiento {tr:.1f} · validación cruzada {cv:.1f}")

<details><summary><b>👀 Ver respuesta</b></summary>

Entrenamiento **68.8** y validación **74.1**: errores **altos y parecidos**. **Subajuste**: con 250 vecinos, el modelo predice casi lo mismo para todos (casi "el promedio"), ignorando las diferencias entre pacientes.
</details>

**Paso 4 ✏️ — La curva de validación.** Dibuja los dos errores para varios valores de k. Ojo: aquí la complejidad va **al revés** (k pequeño = complejo).

In [ ]:
VALORES_K = [1, 3, 5, 10, 15, 20, 30, 50, 100, 200, 250]   # ✏️
resultados = [evaluar_knn(k) for k in VALORES_K]
plt.plot(VALORES_K, [r[0] for r in resultados], "o-", label="Entrenamiento")
plt.plot(VALORES_K, [r[1] for r in resultados], "s-", label="Validación cruzada")
plt.xscale("log"); plt.xlabel("k (← más complejo · más simple →)"); plt.ylabel("RMSE")
plt.legend(); plt.grid(alpha=.3); plt.show()
for k, (a, b) in zip(VALORES_K, resultados):
    print(f"k={k:>3}: entrenamiento {a:5.1f} · validación cruzada {b:5.1f}")

✋ ¿Qué k elegirías? ¿Dónde está el sobreajuste y dónde el subajuste en esta gráfica?

✍️ *Mi k:*

<details><summary><b>👀 Ver respuesta</b></summary>

El mínimo de validación cruzada está en **k = 15** (≈ 55.9), con valores muy parecidos alrededor (k = 10–30). A la **izquierda** (k pequeño) está el sobreajuste: entrenamiento bajísimo, validación alta. A la **derecha** (k grande), el subajuste: los dos errores suben juntos. Es la misma "U" de la Sección 4, **reflejada**.
</details>

**Paso 5 ▶️ — Examen final, una sola vez.** Entrenamos con el k elegido y lo evaluamos con la prueba. Lo comparamos con un modelo "tonto" que siempre predice el **promedio** del entrenamiento (la línea base, Lección 4.7).

In [ ]:
K_ELEGIDO = 15   # ✏️ pon aquí el valor que elegiste

final = make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=K_ELEGIDO)).fit(Xd_tr, yd_tr)
print(f"KNN (k={K_ELEGIDO}) en prueba        : RMSE {rmse(yd_te, final.predict(Xd_te)):.1f}")
print(f"'Siempre el promedio' en prueba: RMSE {rmse(yd_te, np.full(len(yd_te), yd_tr.mean())):.1f}")

<details><summary><b>👀 Ver respuesta</b></summary>

KNN logra **59.0** contra **71.7** del promedio: mejora, pero no es magia. El error de prueba queda razonablemente cerca del de validación cruzada (55.9; con solo 89 pacientes de prueba, unos puntos de diferencia son normales): señal de que **no** hicimos trampa al elegir k. Si el de prueba hubiera salido mucho peor, sospecharíamos que la validación no fue representativa.
</details>

**Paso 6 ✋ — Explícaselo al hospital.** En dos frases, sin jerga: ¿por qué no usaste k = 1, que tenía error **cero**?

✍️ *Mi explicación:*

<details><summary><b>👀 Ver respuesta</b></summary>

Por ejemplo: *"Con k = 1, el modelo copia al paciente más parecido que ya conoce; con los pacientes que ya vio parece perfecto, pero con pacientes nuevos se equivoca mucho porque copia también las rarezas de ese único paciente. Promediando a unos 15 pacientes parecidos, las rarezas se cancelan y las predicciones para pacientes nuevos son más confiables."*
</details>

---
## 📚 Para profundizar
- 📺 **Video:** StatQuest, "Machine Learning Fundamentals: Bias and Variance" (inglés, con subtítulos): el puente a la siguiente lección.
- 📘 **Documentación:** scikit-learn, guía de usuario, *Validation curves: plotting scores to evaluate models* (funciones `validation_curve` y `learning_curve`, que automatizan lo que hicimos a mano).
- 📖 **Libro:** Aurélien Géron, *Hands-On Machine Learning*, capítulo 4, sección *Learning Curves*.

---
⬅️ **Anterior:** [4.3 Umbrales, curva ROC/AUC y curva Precisión-*Recall*](4.3-umbrales-roc-y-precision-recall.ipynb) · ➡️ **Siguiente:** [4.5 El compromiso sesgo-varianza: la analogía del tirador al blanco](4.5-sesgo-varianza.ipynb)